# **Imports**

Below are all the imports used in the **Notebook**.

In [ ]:
# Common
import os 
import keras
import numpy as np 
import pandas as pd
import tensorflow as tf
from IPython.display import clear_output as cls

# Data 
from keras.preprocessing.image import ImageDataGenerator

# Data Visualization
import plotly.express as px
import matplotlib.pyplot as plt

# Model 
from keras.models import Sequential, load_model
from keras.layers import GlobalAvgPool2D as GAP, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Callbacks 
from keras.callbacks import EarlyStopping, ModelCheckpoint

# Pre-Trained Model
from tensorflow.keras.applications import ResNet50, ResNet50V2, InceptionV3, Xception, ResNet152, ResNet152V2

# **Data Info**

Before **loading the data**, let's have a look at the **class distribution**.

In [ ]:
# Class Names
root_path =  "/kaggle/input/culture/Stablediffusion15/Stablediffusion15/Specified/Apparel/"

class_names = sorted(os.listdir(root_path))
n_classes = len(class_names)

# Class Distribution
class_dis = [len(os.listdir(root_path + name)) for name in class_names]


# Show
print(f"Total Number of Classes : {n_classes} \nClass Names : {class_names}")

Let's loot at the **Class Distribution**.

In [ ]:
# Visualize 
fig = px.pie(names=class_names, values=class_dis, title="Class Distribution", hole=0.4)
fig.update_layout({'title':{'x':0.5}})
fig.show()

**Great!!** the classes are **equally distributed**. This ensures that our **model cannot be biased** towards any class.

# **Data Loading**

It's time to **load the data**.

In [ ]:
# Initialize Generator
train_gen = ImageDataGenerator(rescale=1/255., rotation_range=10, validation_split=0.1)

# Load Data
train_ds = train_gen.flow_from_directory(root_path, class_mode='binary', target_size=(224,224), shuffle=True, batch_size=32, subset='training')
valid_ds = train_gen.flow_from_directory(root_path, class_mode='binary', target_size=(224,224), shuffle=True, batch_size=32, subset='validation') 

# **Data Visualization**

The best way to understand the data is to **visualize it**.

In [ ]:
def show_images(GRID=[5,5], model=None, size=(20,20), data=train_ds):
    n_rows = GRID[0]
    n_cols = GRID[1]
    n_images = n_cols * n_rows
    
    i = 1
    plt.figure(figsize=size)
    for images, labels in data:
        id = np.random.randint(len(images))
        image, label = images[id], class_names[int(labels[id])]
        
        plt.subplot(n_rows, n_cols, i)
        plt.imshow(image)
        
        if model is None:
            title = f"Class : {label}"
        else:
            pred = class_names[int(np.argmax(model.predict(image[np.newaxis, ...])))]
            title = f"Org : {label}, Pred : {pred}"
            cls()
        
        plt.title(title)
        plt.axis('off')
        
        i+=1
        if i>=(n_images+1):
            break
            
    plt.tight_layout()
    plt.show()

In [ ]:
show_images()

This is going to be an **Interesting Task**. Let's see how **Transfer Learning** will perform.

# **Model**

In [ ]:
import tensorflow.keras.layers as L
import tensorflow_addons as tfa
# Pre-Trained Model 
base_model = ResNet50V2(input_shape=(224,224,3), include_top=False)
base_model.trainable = False

# Model Architecture
name = "ResNet50V2"
model = Sequential([
        base_model,
        tf.keras.layers.Flatten(),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(128, activation = tfa.activations.gelu),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(64, activation = tfa.activations.gelu),
        tf.keras.layers.Dense(32, activation = tfa.activations.gelu),
        tf.keras.layers.Dense(n_classes, 'softmax')
], name=name)



In [ ]:
learning_rate = 1e-4

optimizer = tfa.optimizers.RectifiedAdam(learning_rate = learning_rate)

model.compile(optimizer = optimizer, 
              loss='sparse_categorical_crossentropy', 
              metrics = ['accuracy'])

STEP_SIZE_TRAIN = train_ds.n // 32
STEP_SIZE_VALID = valid_ds.n // 32



early_stopping_callbacks = tf.keras.callbacks.EarlyStopping(patience = 15, restore_best_weights = True, verbose = 1)

history = model.fit(x = train_ds,
          steps_per_epoch = STEP_SIZE_TRAIN,
          validation_data = valid_ds,
          validation_steps = STEP_SIZE_VALID,
          epochs = 100,
          callbacks = early_stopping_callbacks)

In [ ]:
model.save('ResNet50V2.h5')

In [ ]:
data = pd.DataFrame(history.history)
data.head()

In [ ]:
plt.figure(figsize=(20,5))

plt.subplot(1,2,1)
plt.plot(data.loss, label='Training Loss')
plt.plot(data.val_loss, label='Validation Loss')
plt.title("Loss")
plt.grid()
plt.legend()

plt.subplot(1,2,2)
plt.plot(data.accuracy, label='Training Accuracy')
plt.plot(data.val_accuracy, label='Validation Accuracy')
plt.title("Accuracy")
plt.grid()
plt.legend()

plt.show()

# **Evaluation**

In [ ]:
# Visualize Predictions
show_images(model=model, data=valid_ds)

In [ ]:
# Testing Evaluation
xtest_loss, xtest_acc = model.evaluate(valid_ds)
print(f"Xception Baseline Testing Loss     : {xtest_loss}.")
print(f"Xception Baseline Testing Accuracy : {xtest_acc}.")

In [ ]:
from tabulate import tabulate
import cv2

def show_predictions(model=None, data=None):
    preds = []
    i = 1
    for image in data:
        pred = class_names[int(np.argmax(model.predict(image[np.newaxis, ...])))]
        preds.append(pred)
    
    # Count the occurrences of each prediction
    pred_counts = {pred: preds.count(pred) for pred in set(preds)}
    
    # Calculate the percentage for each prediction
    total_preds = len(preds)
    pred_percentages = {pred: count/total_preds*100 for pred, count in pred_counts.items()}
    
    # Print the prediction percentages as a table
    table = []
    for pred, percentage in pred_percentages.items():
        table.append([pred, f"{percentage:.2f}%"])
    print(tabulate(table, headers=['Prediction', 'Percentage']))
    
    # Create a pie chart
    labels = list(pred_percentages.keys())
    sizes = list(pred_percentages.values())
    fig, ax = plt.subplots()
    ax.pie(sizes, labels=labels, autopct='%1.1f%%')
    ax.axis('equal')
    plt.show()
    
    

# Define the folder path containing the images
folder_path = "/kaggle/input/culture/Stablediffusion15/Stablediffusion15/Predicted/Apparel"

# Create an empty list to store the images
images = []

# Iterate over the files in the folder and load the images
for filename in os.listdir(folder_path):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        image_path = os.path.join(folder_path, filename)
        image = cv2.imread(image_path)
        image = cv2.resize(image, (224, 224))
        images.append(image)

# Convert the list of images to a NumPy array
images = np.array(images)

# Visualize Predictions as a table and pie chart
show_predictions(model=model, data=images)